<a href="https://colab.research.google.com/github/axg1526-create/Weather_Forecaster/blob/main/Kumar_Weather_Forecaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install nodejs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libcares2 libnode109 node-acorn node-busboy node-cjs-module-lexer
  node-undici node-xtend nodejs-doc
Suggested packages:
  npm
The following NEW packages will be installed:
  libcares2 libnode109 node-acorn node-busboy node-cjs-module-lexer
  node-undici node-xtend nodejs nodejs-doc
0 upgraded, 9 newly installed, 0 to remove and 2 not upgraded.
Need to get 16.1 MB of archives.
After this operation, 70.4 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 libcares2 amd64 1.27.0-1.0ubuntu1 [73.7 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 node-xtend all 4.0.2-3 [3,902 B]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 nodejs amd64 18.19.1+dfsg-6ubuntu5 [306 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble/universe amd64 node-acorn all 8.8.1+ds+~cs25.17.7-2 [115

In [ ]:
%%shell
# Install a modern, supported Node.js LTS version (e.g., Node.js 22) in Colab
curl -fsSL https://deb.nodesource.com/setup_24.x | sudo -E bash -
sudo apt-get install -y nodejs
node -v
npm -v

2026-10-13 05:54:27 - Installing pre-requisites
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,382 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/multiverse amd64 Packages [61.7 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,728 kB]
Get:13 http://security.ubuntu.com/ubuntu n

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import pickle
import os

# Load Dataset
df = pd.read_csv('9cf80863-97ea-4ad8-a4ef-6a5f026debbb.csv')

# Preprocess Data
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce')
df = df.dropna(subset=['Date'])

df['day_of_year'] = df['Date'].dt.dayofyear
df['month'] = df['Date'].dt.month
df['year'] = df['Date'].dt.year

# Cyclical encoding for seasonality continuity
df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)

df['Rain'] = pd.to_numeric(df['Rain'], errors='coerce').fillna(0)
df['Temp Max'] = pd.to_numeric(df['Temp Max'], errors='coerce')
df['Temp Min'] = pd.to_numeric(df['Temp Min'], errors='coerce')
df = df.dropna(subset=['Temp Max', 'Temp Min'])

# Train strictly on <= 2023, Test on 2024
train_df = df[df['year'] <= 2023]
test_df = df[df['year'] == 2024]

features = ['doy_sin', 'doy_cos', 'month']
targets = ['Temp Max', 'Temp Min', 'Rain']

os.makedirs('model_artifacts', exist_ok=True)
print("--- Training Models (1951-2023) & Evaluating on 2024 ---")

for target in targets:
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(train_df[features], train_df[target])

    # Save model artifact
    model_filename = f"model_artifacts/{target.lower().replace(' ', '_')}_model.pkl"
    with open(model_filename, 'wb') as f:
        pickle.dump(model, f)

    # Evaluate on 2024 test set
    preds = model.predict(test_df[features])
    mse = mean_squared_error(test_df[target], preds)
    print(f"MSE for {target} in 2024: {mse:.4f}")

print("\nTraining complete. Models saved successfully in /model_artifacts.")

EmptyDataError: No columns to parse from file

In [ ]:
%%writefile predict.py
import sys
import pickle
import numpy as np
import pandas as pd
import json
from datetime import datetime

def predict_weather(date_str):
    try:
        dt = datetime.strptime(date_str.strip(), '%d-%m-%y')
    except ValueError as e:
        print(json.dumps({"error": f"Invalid date format. Use DD-MM-YY. Details: {str(e)}"}))
        sys.exit(1)

    day_of_year = dt.timetuple().tm_yday
    month = dt.month

    doy_sin = np.sin(2 * np.pi * day_of_year / 365.25)
    doy_cos = np.cos(2 * np.pi * day_of_year / 365.25)

    X_input = pd.DataFrame([[doy_sin, doy_cos, month]], columns=['doy_sin', 'doy_cos', 'month'])

    results = {}
    targets = ['Temp Max', 'Temp Min', 'Rain']

    for target in targets:
        model_filename = f"model_artifacts/{target.lower().replace(' ', '_')}_model.pkl"
        with open(model_filename, 'rb') as f:
            model = pickle.load(f)
        pred = model.predict(X_input)[0]
        results[target] = float(pred)

    print(json.dumps(results))

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print(json.dumps({"error": "Date argument missing."}))
        sys.exit(1)
    predict_weather(sys.argv[1])

Overwriting predict.py


In [ ]:
%%writefile index.js
const { spawn } = require('child_process');
const path = require('path');

// Get date from command line argument (e.g., node index.js 25-12-26)
const inputDate = process.argv[2];

if (!inputDate) {
    console.error("Please provide a date in DD-MM-YY format. Example: node index.js 25-12-26");
    process.exit(1);
}

const pythonScript = path.join(__dirname, 'predict.py');
const pythonProcess = spawn('python3', [pythonScript, inputDate]);

let outputData = '';
let errorData = '';

pythonProcess.stdout.on('data', (data) => {
    outputData += data.toString();
});

pythonProcess.stderr.on('data', (data) => {
    errorData += data.toString();
});

pythonProcess.on('close', (code) => {
    if (code !== 0 || errorData) {
        console.error('Error during prediction:', errorData || outputData);
        return;
    }

    try {
        const result = JSON.parse(outputData.trim());
        if (result.error) {
            console.log(`Error: ${result.error}`);
        } else {
            console.log(`\n--- Weather Forecast for ${inputDate} ---`);
            console.log(`* Maximum Temperature : ${result['Temp Max'].toFixed(2)} °C`);
            console.log(`* Minimum Temperature : ${result['Temp Min'].toFixed(2)} °C`);
            console.log(`* Rainfall            : ${result['Rain'].toFixed(2)} mm`);
            console.log('-----------------------------------------');
        }
    } catch (err) {
        console.error('Failed to parse model output:', outputData);
    }
});

Overwriting index.js


In [ ]:
!node index.js 25-12-27

node:internal/modules/cjs/loader:1215
  throw err;
  ^

Error: Cannot find module '/content/index.js'
    at Module._resolveFilename (node:internal/modules/cjs/loader:1212:15)
    at Module._load (node:internal/modules/cjs/loader:1043:27)
    at Function.executeUserEntryPoint [as runMain] (node:internal/modules/run_main:164:12)
    at node:internal/main/run_main_module:28:49 {
  code: 'MODULE_NOT_FOUND',
  requireStack: []
}

Node.js v20.19.0


In [ ]:
%%shell
curl -fsSL https://deb.nodesource.com/setup_24.x | sudo -E bash -
sudo apt-get install -y nodejs
npm install -g express localtunnel
node -v

2026-10-13 05:46:57 - Installing pre-requisites
Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:6 https://deb.nodesource.com/node_24.x nodistro InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:8 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
30 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package list

In [ ]:
%%writefile server.js
const express = require('express');
const { spawn } = require('child_process');
const path = require('path');

const app = express();
const PORT = 3000;

app.use(express.urlencoded({ extended: true }));
app.use(express.json());

// Serve HTML UI
app.get('/', (req, res) => {
    res.send(`
        <!DOCTYPE html>
        <html lang="en">
        <head>
            <meta charset="UTF-8">
            <meta name="viewport" content="width=device-width, initial-scale=1.0">
            <title>Bangalore Weather Predictor</title>
            <style>
                body { font-family: Arial, sans-serif; background-color: #f4f7f6; margin: 0; padding: 50px; display: flex; justify-content: center; }
                .card { background: white; padding: 30px; border-radius: 10px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); width: 400px; text-align: center; }
                h2 { color: #333; }
                input { width: 80%; padding: 10px; margin: 15px 0; border: 1px solid #ccc; border-radius: 5px; font-size: 16px; text-align: center; }
                button { background: #007bff; color: white; border: none; padding: 10px 20px; font-size: 16px; border-radius: 5px; cursor: pointer; width: 85%; }
                button:hover { background: #0056b3; }
                #result { margin-top: 20px; text-align: left; background: #e9ecef; padding: 15px; border-radius: 5px; display: none; }
                .error { color: red; font-weight: bold; }
            </style>
        </head>
        <body>
            <div class="card">
                <h2>Bangalore Weather Predictor</h2>
                <p>Enter a date after 2024 (Format: DD-MM-YY)</p>
                <input type="text" id="dateInput" placeholder="e.g., 25-12-26">
                <br>
                <button onclick="getPrediction()">Predict Weather</button>
                <div id="result"></div>
            </div>

            <script>
                async function getPrediction() {
                    const dateVal = document.getElementById('dateInput').value;
                    const resultDiv = document.getElementById('result');
                    resultDiv.style.display = 'block';
                    resultDiv.innerHTML = "Predicting...";

                    try {
                        const response = await fetch('/predict', {
                            method: 'POST',
                            headers: { 'Content-Type': 'application/json' },
                            body: JSON.stringify({ date: dateVal })
                        });
                        const data = await response.json();

                        if (data.error) {
                            resultDiv.innerHTML = \`<span class="error">Error: \${data.error}</span>\`;
                        } else {
                            resultDiv.innerHTML = \`
                                <strong>Forecast for \${dateVal}:</strong><br>
                                🌡️ Max Temp: <b>\${data['Temp Max'].toFixed(2)} °C</b><br>
                                🌡️ Min Temp: <b>\${data['Temp Min'].toFixed(2)} °C</b><br>
                                🌧️ Rainfall: <b>\${data['Rain'].toFixed(2)} mm</b>
                            \`;
                        }
                    } catch (err) {
                        resultDiv.innerHTML = '<span class="error">Failed to connect to server.</span>';
                    }
                }
            </script>
        </body>
        </html>
    `);
});

// Prediction API Endpoint
app.post('/predict', (req, res) => {
    const inputDate = req.body.date;
    if (!inputDate) {
        return res.json({ error: "Date is required." });
    }

    const pythonScript = path.join(__dirname, 'predict.py');
    const pythonProcess = spawn('python3', [pythonScript, inputDate]);

    let outputData = '';
    let errorData = '';

    pythonProcess.stdout.on('data', (data) => { outputData += data.toString(); });
    pythonProcess.stderr.on('data', (data) => { errorData += data.toString(); });

    pythonProcess.on('close', (code) => {
        if (code !== 0 || errorData) {
            return res.json({ error: errorData || "Python execution error." });
        }
        try {
            const result = JSON.parse(outputData.trim());
            res.json(result);
        } catch (err) {
            res.json({ error: "Failed to parse model output." });
        }
    });
});

app.listen(PORT, () => {
    console.log(`Server running on port ${PORT}`);
});

Overwriting server.js
